# 03 — Fine-tuning LoRA con PEFT

**Level 4 — Model Ops & Optimization**

Entrenamos un adaptador LoRA sobre SmolLM2-360M con el dataset del agente.
El notebook permite ver el loss bajando paso a paso — la evidencia de que
el modelo está aprendiendo.

In [1]:
import json
import time
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

MODELO_BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"
RUTA_DATASET = Path("../data/finetune_dataset.jsonl")
RUTA_ADAPTER = Path("../adapters/lora")
MAX_PASOS = 40
MAX_LONGITUD = 512

/workspaces/student-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cargar y tokenizar el dataset

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ejemplos = [
    json.loads(linea)
    for linea in RUTA_DATASET.read_text(encoding="utf-8").splitlines()
    if linea.strip()
]
textos = [
    tokenizer.apply_chat_template(e["messages"], tokenize=False)
    for e in ejemplos
]
dataset = Dataset.from_dict({"text": textos})

def tokenizar(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LONGITUD)

dataset = dataset.map(tokenizar, batched=True, remove_columns=["text"])
print(f"{len(dataset)} ejemplos tokenizados")

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map: 100%|██████████| 8/8 [00:00<00:00, 1781.12 examples/s]

8 ejemplos tokenizados


## Configurar LoRA (0.23% de los parámetros)

In [3]:
modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float32)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo.to(dispositivo)
modelo.config.use_cache = False
total_params = sum(p.numel() for p in modelo.parameters())
print(f"{total_params / 1e6:.0f}M parametros en el modelo base")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
modelo = get_peft_model(modelo, config)
entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"parametros entrenables: {entrenables:,} ({100 * entrenables / total_params:.2f}% del total)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  97%|█████████▋| 280/290 [00:00<00:00, 2793.45it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2751.05it/s]

362M parametros en el modelo base
parametros entrenables: 819,200 (0.23% del total)


## Entrenar: el loss bajando

In [4]:
args = TrainingArguments(
    output_dir="checkpoints",
    max_steps=MAX_PASOS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    use_cpu=(dispositivo == "cpu"),
)
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
)
trainer.train()

Step,Training Loss
10,2.487140
20,2.305013
30,2.252440
40,2.092938


TrainOutput(global_step=40, training_loss=2.2843830585479736, metrics={'train_runtime': 8.6648, 'train_samples_per_second': 9.233, 'train_steps_per_second': 4.616, 'total_flos': 24283679712000.0, 'train_loss': 2.2843830585479736, 'epoch': 10.0})

## Guardar el adaptador

In [5]:
RUTA_ADAPTER.mkdir(parents=True, exist_ok=True)
modelo.save_pretrained(str(RUTA_ADAPTER))
tokenizer.save_pretrained(str(RUTA_ADAPTER))
print(f"Adaptador guardado en {RUTA_ADAPTER}")

Adaptador guardado en ../adapters/lora


## Conclusión

- El **loss baja** paso a paso: 2.30 → ~2.0 en 40 pasos (evidencia de aprendizaje)
- Solo **0.23%** de los parámetros se entrenaron (819K de 362M)
- El adaptador pesa unos **pocos MB** — el modelo base queda intacto